![](https://storage.googleapis.com/mle-courses-prod/users/61b6fa1ba83a7e37c8309756/private-files/101f69f0-f00d-11ef-9acf-6f7079495a92-LLMs__2_.png)

 [Slide chi tiết DPO + RLHF](https://drive.google.com/file/d/1o2nmBN_aPPzum9FUoxdUDZ7GoDDpJ_cF/view?usp=drive_link)

### 1. Load dữ liệu

In [1]:
#!gdown 19h1Tvntu9vzeku6Zq3Q2kx1lFXbfi5le

In [2]:
import json
import os
import urllib

# Define the file path
file_path = "vi-instruction-data-with-preference.json"

# Check if the file exists
if os.path.exists(file_path):
    # Load JSON file
    with open(file_path, "r", encoding="utf-8") as json_file:
        data = json.load(json_file)

    # Print the loaded data
    print(json.dumps(data, indent=4, ensure_ascii=False))
else:
    print(f"File '{file_path}' not found!")



[
    {
        "instruction": "Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần lượt là 6 cm và 8 cm.",
        "input": "",
        "output": "Cạnh huyền của tam giác là 10 cm.",
        "rejected": "Cạnh huyền của tam giác là 10 cm, đấy.",
        "chosen": "Cạnh huyền của tam giác là 10 cm."
    },
    {
        "instruction": "Kể tên ba loài động vật hoạt động vào ban ngày.",
        "input": "",
        "output": "1. Sóc\n2. Đại bàng\n3. Hổ",
        "chosen": "1. Sóc\n2. Đại bàng\n3. Hổ",
        "rejected": "1. Sóc\n2. Đại bàng\n3. Hổ"
    },
    {
        "instruction": "Viết lại câu sau theo cách trang trọng hơn: \"Tôi cần tìm ra vấn đề là gì.\"",
        "input": "",
        "output": "Tôi cần xác định vấn đề là gì.",
        "chosen": "Tôi cần xác định vấn đề một cách rõ ràng.",
        "rejected": "Tôi cần xác định vấn đề là gì."
    },
    {
        "instruction": "Xác định thán từ trong câu sau.",
        "input": "Wow, màn trình diễn thật ấn tượng!",
       

In [4]:
import torch
from torch.utils.data import Dataset


class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            prompt = format_input(entry) # type: ignore
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [6]:
import pprint

pprint.pp(data[2])


{'instruction': 'Viết lại câu sau theo cách trang trọng hơn: "Tôi cần tìm ra '
                'vấn đề là gì."',
 'input': '',
 'output': 'Tôi cần xác định vấn đề là gì.',
 'chosen': 'Tôi cần xác định vấn đề một cách rõ ràng.',
 'rejected': 'Tôi cần xác định vấn đề là gì.'}


In [7]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [9]:
model_input = format_input(data[2])
print(model_input)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Viết lại câu sau theo cách trang trọng hơn: "Tôi cần tìm ra vấn đề là gì."


In [10]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [11]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"
):
    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []

    }

    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # Process each item in the batch
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # Adjust padding according to the common maximum length
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # Set mask for all padding tokens to False
            mask[len(sequence):] = False

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # Final processing
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # Stack all sequences into a tensor for the given key
        tensor_stack = torch.stack(batch_data[key])

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # Move to the specified device
        batch_data[key] = tensor_stack.to(device)

    return batch_data

In [12]:
import torch

from functools import partial

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)

Device: cuda


In [13]:
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)


{'instruction': 'Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần '
                'lượt là 6 cm và 8 cm.',
 'input': '',
 'output': 'Cạnh huyền của tam giác là 10 cm.',
 'rejected': 'Cạnh huyền của tam giác là 10 cm, đấy.',
 'chosen': 'Cạnh huyền của tam giác là 10 cm.'}

{'instruction': 'Kể tên ba loài động vật hoạt động vào ban ngày.',
 'input': '',
 'output': '1. Sóc\n2. Đại bàng\n3. Hổ',
 'chosen': '1. Sóc\n2. Đại bàng\n3. Hổ',
 'rejected': '1. Sóc\n2. Đại bàng\n3. Hổ'}


In [14]:
%pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.0 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [15]:
import tiktoken
from torch.utils.data import DataLoader


tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

In [16]:
for batch in example_dataloader:
    break

print("batch.keys:", batch.keys())

batch.keys: dict_keys(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])


In [17]:
batch["prompt"]

[tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    51, 39588,    71,   269,   157,   118,
            94,    77,    71,   289,  4669,   157,   119,   223,    77,   269,
           157,   119,   100,    64, 21885,   308,    72,  6557,    66,   410,
            84, 27083,   782,   269, 10205,   387,    72,   269,   157,   118,
            94,    77,    71,   308, 10205,    66,   410,    84, 27083,   782,
           300,   157,   118,   100,    77,   300,   130,   108,   157,   119,
            96,    83,   300, 24247,   718, 12067,   410, 24247,   807, 12067,
            13]),
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    42,   157,   119,   225,   256, 25792,
            77, 26605,  2376, 2424

In [18]:
batch["chosen"]

tensor([[21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    51, 39588,    71,   269,   157,   118,
            94,    77,    71,   289,  4669,   157,   119,   223,    77,   269,
           157,   119,   100,    64, 21885,   308,    72,  6557,    66,   410,
            84, 27083,   782,   269, 10205,   387,    72,   269,   157,   118,
            94,    77,    71,   308, 10205,    66,   410,    84, 27083,   782,
           300,   157,   118,   100,    77,   300,   130,   108,   157,   119,
            96,    83,   300, 24247,   718, 12067,   410, 24247,   807, 12067,
            13,   198,   198, 21017, 18261,    25,   198,    34,   157,   118,
            94,    77,    71,   289,  4669,   157,   119,   223,    77,   269,
           157,   119,   100,    64, 21885,   308,    72,  6557,    66,   300,
         24247,   838, 12067,    13, 50256, 50256, 5

In [19]:
def decode_tokens_from_batch(token_ids, tokenizer):
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

In [20]:
text = decode_tokens_from_batch(
    token_ids=batch["prompt"][0],  # [0] for the first entry in the batch
    tokenizer=tokenizer,
)
print(text)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần lượt là 6 cm và 8 cm.


In [21]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0],  # [0] for the first entry in the batch
    tokenizer=tokenizer,
)
print(text)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần lượt là 6 cm và 8 cm.

### Response:
Cạnh huyền của tam giác là 10 cm.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [22]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0],  # [0] for the first entry in the batch
    tokenizer=tokenizer,
)
print(text)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần lượt là 6 cm và 8 cm.

### Response:
Cạnh huyền của tam giác là 10 cm, đấy.<|endoftext|>


In [23]:
print("chosen inputs:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)

chosen inputs: torch.Size([132])
chosen mask:   torch.Size([132])


In [24]:
batch["chosen"][0]

tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198,    51, 39588,    71,   269,   157,   118,
           94,    77,    71,   289,  4669,   157,   119,   223,    77,   269,
          157,   119,   100,    64, 21885,   308,    72,  6557,    66,   410,
           84, 27083,   782,   269, 10205,   387,    72,   269,   157,   118,
           94,    77,    71,   308, 10205,    66,   410,    84, 27083,   782,
          300,   157,   118,   100,    77,   300,   130,   108,   157,   119,
           96,    83,   300, 24247,   718, 12067,   410, 24247,   807, 12067,
           13,   198,   198, 21017, 18261,    25,   198,    34,   157,   118,
           94,    77,    71,   289,  4669,   157,   119,   223,    77,   269,
          157,   119,   100,    64, 21885,   308,    72,  6557,    66,   300,
        24247,   838, 12067,    13, 50256, 50256, 50256, 50256, 

### Giải thích về mask

**chosen_mask:** chọn lọc xác suất output của mô hình $p(y^w|x)$ để chỉ chọn các ID token tương ứng với phản hồi trong chuỗi token truyền vào model, tức là loại bỏ tất cả các token của prompt và padding.

Trong chuỗi
```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Tính cạnh huyền của tam giác vuông có hai cạnh góc vuông lần lượt là 6 cm và 8 cm.

### Response:
Cạnh huyền của tam giác là 10 cm, đấy.<|endoftext|>
```

Thì chosen_mask sẽ đánh **True** với chuỗi con chứa output

```
### Response:
Cạnh huyền của tam giác là 10 cm, đấy.
```

Còn **False** với phần còn lại (Kể cả padding)


**Mục đích**: Để khi đưa $x$ qua model sẽ được kết quả và lọc xác suất $p(y^w|x)$ từ kết quả đó.

In [26]:
batch["chosen_mask"][0]

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False, False, False, False, 

In [27]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

### Response:
Cạnh huyền của tam giác là 10 cm.


Reject mask cũng tương tự như vậy

In [28]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

### Response:
Cạnh huyền của tam giác là 10 cm, đấy.


In [29]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [30]:
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [32]:
print("Train loader:")
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape,
    )

Train loader:
torch.Size([8, 122]) torch.Size([8, 122])
torch.Size([8, 157]) torch.Size([8, 157])
torch.Size([8, 144]) torch.Size([8, 144])
torch.Size([8, 167]) torch.Size([8, 167])


### 2. Lập trình mất mát của DPO

$$J_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y^w,y^l) \sim D} \left[ \log \sigma \left( \beta \log \frac{p_{\theta}^{RL}(y^w \mid x)}{p_{\theta}^{PT}(y^w \mid x)} - \beta \log \frac{p_{\theta}^{RL}(y^l \mid x)}{p_{\theta}^{PT}(y^l \mid x)} \right) \right]
$$

Tương đương


$$
J_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y^w,y^l) \sim D} \left[ \log \sigma \left( \beta \left( \log p_{\theta}^{RL}(y^w | x) - \log p_{\theta}^{PT}(y^w | x) \right) - \beta \left( \log p_{\theta}^{RL}(y^l | x) - \log p_{\theta}^{PT}(y^l | x) \right) \right) \right]
$$


Tương đương


$$
J_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y^w,y^l) \sim D} \left[ \log \sigma \left( \beta \left( \log p_{\theta}^{RL}(y^w | x) - \log  p_{\theta}^{RL}(y^l | x)  \right) - \beta \left( \log p_{\theta}^{PT}(y^w | x) ) - \log p_{\theta}^{PT}(y^l | x) \right) \right) \right]
$$


In [33]:
import torch.nn.functional as F

def compute_dpo_loss(
      model_chosen_logprobs,
      model_rejected_logprobs,
      reference_chosen_logprobs,
      reference_rejected_logprobs,
      beta=0.1,
    ):
    """Compute the DPO loss for a batch of policy and reference model log probabilities.

    Args:
        policy_chosen_logprobs: Log probabilities of the policy model for the chosen responses. Shape: (batch_size,)
        policy_rejected_logprobs: Log probabilities of the policy model for the rejected responses. Shape: (batch_size,)
        reference_chosen_logprobs: Log probabilities of the reference model for the chosen responses. Shape: (batch_size,)
        reference_rejected_logprobs: Log probabilities of the reference model for the rejected responses. Shape: (batch_size,)
        beta: Temperature parameter for the DPO loss; typically something in the range of 0.1 to 0.5. We ignore the reference model as beta -> 0.
        label_smoothing: conservativeness for DPO loss.

    Returns:
        A tuple of three tensors: (loss, chosen_rewards, rejected_rewards).
    """

    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    logits = model_logratios - reference_logratios

    # DPO (Eq. 7 of https://arxiv.org/pdf/2305.18290.pdf)
    losses = -F.logsigmoid(beta * logits)

    # Optional values to track progress during training
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    # .mean() to average over the samples in the batch
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

In [34]:
def compute_logprobs(logits, labels, selection_mask=None):
    """
    Compute log probabilities.

    Args:
      logits: Tensor of shape (batch_size, num_tokens, vocab_size)
      labels: Tensor of shape (batch_size, num_tokens)
      selection_mask: Tensor for shape (batch_size, num_tokens)

    Returns:
      mean_log_prob: Mean log probability excluding padding tokens.
    """

    # Labels are the inputs shifted by one
    labels = labels[:, 1:].clone()

    # Truncate logits to match the labels num_tokens
    logits = logits[:, :-1, :]

    log_probs = F.log_softmax(logits, dim=-1)

    # Gather the log probabilities for the actual labels
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # Lọc kết quả
    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()

        # Apply the mask to filter out padding tokens
        selected_log_probs = selected_log_probs * mask

        # Calculate the average log probability excluding padding tokens
        # This averages over the tokens, so the shape is (batch_size, num_tokens)
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)

        return avg_log_prob

    else:
        return selected_log_probs.mean(-1)

In [35]:
# Sample data
logits = torch.tensor(
    [[2.0, 1.0, 0.1],
     [0.5, 2.5, 0.3]])  # Shape: (2, 3)
targets = torch.tensor([0, 2])  # Shape: (2,)


# Manual loss using torch.gather
log_softmax_logits = F.log_softmax(logits, dim=1)  # Shape: (2, 3)
selected_log_probs = torch.gather(
    input=log_softmax_logits,
    dim=1,
    index=targets.unsqueeze(1), # Shape 2, 1
).squeeze(1)  # Shape: (2,)
manual_loss = -selected_log_probs.mean()  # Averaging over the batch


# PyTorch loss
cross_entropy_loss = F.cross_entropy(logits, targets)

print(manual_loss, cross_entropy_loss)

tensor(1.4185) tensor(1.4185)


In [36]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """Compute the DPO loss on an input batch"""

    # where policy_model(batch["chosen"]) are the logits
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]).logits,
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]).logits,
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )

    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]).logits,
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]).logits,
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

### 3. Load mô hình

In [38]:

import torch
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler

# Load LLaMA 3.2 model and tokenizer
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Model sẽ học
policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
).to(device)

# Model sft ban đầu, không học
reference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Freeze reference model (no gradient updates)
for param in reference_model.parameters():
    param.requires_grad = False

# Optimizer & Learning Rate Scheduler
optimizer = AdamW(policy_model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # Assuming 3 epochs
lr_scheduler = get_scheduler("cosine", optimizer=optimizer, num_warmup_steps=100, num_training_steps=num_training_steps)

# Training Loop
num_epochs = 10
beta = 0.1  # DPO temperature parameter
policy_model.train()

for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()

        loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(batch, policy_model, reference_model, beta)

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

    # Validation Step
    policy_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_batch in val_loader:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(val_batch, policy_model, reference_model, beta)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}")

    # Save checkpoint every epoch
    policy_model.save_pretrained(f"llama3_dpo_epoch{epoch+1}")
    tokenizer.save_pretrained(f"llama3_dpo_epoch{epoch+1}")

    policy_model.train()  # Set model back to training mode

print("Training completed!")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
401 Client Error. (Request ID: Root=1-686ba0e5-76b315f45751e9392785584c;3fe5adeb-3760-4e1a-9e8f-2b6990811ada)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
def generate_answer(prompt, model, tokenizer, max_length=512, temperature=0.7, top_p=0.9):
    """Generates an answer from the fine-tuned policy model."""
    model.eval()

    tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example usage
prompt = """

### Instruction:
Viết lại câu sau bằng giọng chủ động.

### Input:
Bức thư đã được viết bởi anh ấy.

"""

response = generate_answer(prompt, reference_model, tokenizer)
print(response)



### Instruction:
Viết lại câu sau bằng giọng chủ động.

### Input:
Bức thư đã được viết bởi anh ấy.

### Output:
Bức thư đã được viết bằng giọng chủ động. 

### Ví dụ:
Nếu ta muốn viết lại một bức thư đã được viết, ta cần thay thế từ "by" với từ "did". Ví dụ:

- "Tôi đã viết thư này cho bạn." -> "Bức thư này đã được tôi viết."

- "Anh ấy đã viết thư này." -> "Bức thư này đã được anh ấy viết."

- "Tôi đã viết thư này để bạn biết." -> "Bức thư này đã được tôi viết để bạn biết."

### Lưu ý:
Giọng chủ động là khi người viết thay đổi hướng ngữ của từ "by", từ "to" hoặc từ "for" để thay thế bằng từ "did", từ "did to" hoặc từ "did for" để thể hiện rằng người viết đã thực hiện một hành động chủ động.


### Nguồn tham khảo
[DPO với GPT](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/04_preference-tuning-with-dpo/dpo-from-scratch.ipynb)
   